# 📄 Multi-Document PDF RAG Assistant

**Tech Stack:** pypdf · sentence-transformers · FAISS · TinyLlama-1.1B-Chat


## 0. Runtime Check
Pastikan GPU aktif: **Runtime → Change runtime type → T4 GPU**

In [35]:
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4


## 1. Install Dependencies

In [36]:
!pip install pypdf bitsandbytes faiss-cpu
print('✅ Dependencies installed')

✅ Dependencies installed


## 2. Clone Repository & Setup Structure

In [ ]:
import os
import shutil
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
github_token = user_secrets.get_secret("GITHUB_TOKEN")

github_username = "rakdaf08"
repo_name = "nolimit-ds-test-rakadaffa"

if os.path.exists(repo_name):
    shutil.rmtree(repo_name)
    print(f"Folder {repo_name} berhasil dihapus.")

!git clone https://{github_token}@github.com/{github_username}/{repo_name}.git

%cd {repo_name}

Cloning into 'nolimit-ds-test-rakadaffa'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 25 (delta 6), reused 23 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 18.04 KiB | 3.01 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/kaggle/working/nolimit-ds-test-rakadaffa/nolimit-ds-test-rakadaffa/nolimit-ds-test-rakadaffa


## 3. Dataset Exploration (Phase 1 Task 2)

In [40]:
import sys, json
sys.path.insert(0, 'src')
from loader import get_dataset_summary
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

summary = get_dataset_summary('/kaggle/input/datasets/manisha717/dataset-of-pdf-files/Pdf')
print(f"Total PDFs  : {summary['total_files']}")
print(f"Total pages : {summary['total_pages']}")
print()
df = pd.DataFrame(summary['files'])
display(df)

parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams
parsing for Object Streams
invalid pdf header: b'\r\n\r\n\r'
CAUTION: startxref found while searching for %%EOF. The file might be truncated and some data might not be read.
EOF marker not found
Ignoring wrong pointing object 1 65536 (offset 0)
Ignoring wrong pointing object 15 65536 (offset 0)
incorrect startxref pointer(3)
incorrect startxref pointer(3)
parsing for Object Streams
incorrect startxref pointer(3)
incorrect startxref pointer(4)
parsing for Object Streams
invalid pdf header: b'\n<!DO'
EOF marker not found
Ignoring wrong pointing object 2 65536 (offset 0)
Ignoring wrong pointing object 35 65536 (offset 0)
Ignoring wrong pointing object 55 65536 (offset 0)
Ignoring wrong pointing object 74 65536 (offset 0)
Ignoring wrong pointing object 92 65536 (offset 0)
Ignoring wrong pointing object 112 65536 (offset 0)
Ignoring wrong pointing object 139 65536 (offset 0)
Ignoring wrong pointing object 1

Total PDFs  : 1076
Total pages : 13380



,filename,size_kb,pages,readable
0,08036c5a50a93da84c5c45ba468c58159d75281e.pdf,391.71,5,True
1,0a29925ccc5e6299e132a73325956a3abef6dd26.pdf,55.83,2,True
2,0e21835a42a6df2405496f62647058ff855743c1.pdf,1194.53,5,True
3,11613a97cef51ad28635fdd86915e74d94cff227.pdf,216.83,5,True
4,12851f0053449570257ff3dfe552621a8dd63d53.pdf,404.51,6,True
...,...,...,...,...
1071,f415cd9cf7bdca0a6a1fde12e0930661927d5b3b.pdf,413.29,5,True
1072,f55a8d69c6047f9037f79225c367766987e25f2b.pdf,206.78,4,True
1073,f997183fa6bf1ef75e31406d98f025c16e527d20.pdf,613.90,49,True
1074,fb8bb0cefab7494ca8c56a4b5b332922e404e465.pdf,358.94,8,True


## 4. Indexing Pipeline

In [ ]:
import torch, sys
sys.path.insert(0, 'src')

from loader import load_pdfs
from chunker import chunk_documents
from embedder import Embedder
from vectorstore import VectorStore
from retriever import Retriever

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# Load PDFs
documents = load_pdfs('/kaggle/input/datasets/manisha717/dataset-of-pdf-files/Pdf')

# Chunk
chunks = chunk_documents(documents, chunk_size=500, chunk_overlap=100)

# Embed
embedder = Embedder(device=DEVICE)
embeddings = embedder.encode_chunks(chunks)

# Build FAISS index
vector_store = VectorStore(embedder.embedding_dim)
vector_store.add(embeddings, chunks)
vector_store.save('faiss_index')

# Build retriever
retriever = Retriever(embedder, vector_store)
print('\n✅ Indexing pipeline complete')

Device: cuda
Found 1076 PDF file(s) in '/kaggle/input/datasets/manisha717/dataset-of-pdf-files/Pdf'


Loading PDFs:   4%|▍         | 44/1076 [00:53<05:58,  2.88it/s]  incorrect startxref pointer(1)
parsing for Object Streams
Object 47 0 found
Loading PDFs:   6%|▌         | 62/1076 [01:00<07:05,  2.38it/s]invalid pdf header: b'\r\n\r\n\r'
CAUTION: startxref found while searching for %%EOF. The file might be truncated and some data might not be read.
EOF marker not found


  [WARN] Failed to read '3P5D3UKXU2R6I2TK4OJSLL6LGIQJ4NY5.pdf': Stream has ended unexpectedly


Loading PDFs:   7%|▋         | 76/1076 [01:25<30:24,  1.82s/it]  Ignoring wrong pointing object 1 65536 (offset 0)
Ignoring wrong pointing object 15 65536 (offset 0)
Loading PDFs:   9%|▊         | 93/1076 [01:28<04:51,  3.38it/s]

  [WARN] Failed to read '4GJGAIUVBMLM3W7O5SV4EKDNKC4DVOCL.pdf': File has not been decrypted


...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 384


Batches:   0%|          | 0/1275 [00:00<?, ?it/s]

Generated 81539 embeddings of dim 384
VectorStore: 81539 vector(s) indexed
VectorStore saved → faiss_index

✅ Indexing pipeline complete


## 5. Load TinyLlama LLM

In [42]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline as hf_pipeline

LLM_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Loading {LLM_MODEL} on {DEVICE}...')

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    device_map='auto' if DEVICE == 'cuda' else None,
)
if DEVICE == 'cpu':
    llm = llm.to('cpu')

generator = hf_pipeline(
    'text-generation',
    model=llm,
    tokenizer=tokenizer,
    device=None if DEVICE == 'cuda' else -1,
)
print('✅ LLM loaded')

Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0 on cuda...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ LLM loaded


## 6. RAG Query Function

In [43]:
FALLBACK = 'I could not find relevant information in the document collection.'

PROMPT_TEMPLATE = """<|system|>
You are a helpful assistant. Answer questions strictly using the provided context.
If the answer is not in the context, say: \"{fallback}\"
</s>
<|user|>
Context:
{context}

Question: {question}
</s>
<|assistant|>
"""

def rag_query(question, top_k=5):
    results  = retriever.retrieve(question, top_k=top_k)
    context  = retriever.format_context(results)
    citations = retriever.format_citations(results)

    prompt = PROMPT_TEMPLATE.format(
        fallback=FALLBACK, context=context, question=question
    )

    output = generator(
        prompt,
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

    full_text = output[0]['generated_text']
    answer    = full_text[len(prompt):].strip()

    return {
        'question': question,
        'answer':   answer or FALLBACK,
        'sources':  citations,
    }

print('✅ rag_query() ready')

✅ rag_query() ready


## 7. Run a Test Query

In [ ]:
QUESTION = 'What is the main topic discussed in the documents?'

result = rag_query(QUESTION, top_k=5)

print('=' * 60)
print(f'QUESTION: {result["question"]}')
print('=' * 60)
print(f'ANSWER:\n{result["answer"]}')
print()
print('SOURCES:')
for src in result['sources']:
    print(f"  [{src['source_num']}] {src['filename']} — page {src['page_number']} (score={src['score']})")
    print(f"      {src['snippet'][:150]}...")
    print()

Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION: What is the main topic discussed in the documents?
ANSWER:
The main topic discussed in the documents is the relevance of documents.

SOURCES:
  [1] LCFQJGJLCOJ56B3YM3XIPRJ7DFUQPTDG.pdf — page 96 (score=0.6223)
      the relevant 
documents....

  [2] NKJZFXDHGHWNNHGULXM5PSPE7GWWUTR2.pdf — page 121 (score=0.5676)
      - 115 -
 
 
 
 
 
 
 
APPENDIX B - MEMORANDUM OF UNDERSTANDING...

  [3] SYBT26QCMSRHEOFZDZM4TZ5CCJ6T34MF.pdf — page 53 (score=0.5491)
      ns the following sections: 
Issue: A brief question framing the issue or topic. 
Issue Description: A description of the issue or topic plus (if appro...

  [4] 4UCZ4AEHAQQE2MYSFOGR6GRRPKNIAFCW.pdf — page 1 (score=0.5103)
      n 10.)
X
Is the document necessary to protect or safeguard the health, welfare (budget
levels necessary to provide services to the citizens of the sta...

  [5] MMSNF4WV7XLHQFKEQYKHHH7GJPPQJQ7U.pdf — page 33 (score=0.5088)
      LaThuile Italy / David Finley, Fermilab / March 2007            Slide 33

## 8. Evaluation

In [ ]:
import pandas as pd

EVAL_QUESTIONS = [
    'What is the main topic discussed in the first document?',
    'Summarize the key points of the document collection.',
    'What specific information is provided about any technical topic?',
    'What conclusions or recommendations are made?',
    'Are there any data, statistics, or numbers mentioned?',
]

rows = []
for q in EVAL_QUESTIONS:
    r = rag_query(q, top_k=5)
    top_src = r['sources'][0] if r['sources'] else {}
    rows.append({
        'question':          q,
        'answer_preview':    r['answer'][:200],
        'top_source':        top_src.get('filename', '-'),
        'top_page':          top_src.get('page_number', '-'),
        'top_score':         top_src.get('score', 0),
        'has_fallback':      r['answer'] == 'I could not find relevant information in the document collection.',
    })

df_eval = pd.DataFrame(rows)
display(df_eval)

# Save results
import os                                                                                                                                                
os.makedirs('outputs', exist_ok=True)                                                                                                                    

df_eval.to_markdown('outputs/evaluation_results.md', index=False)                                                                                        
print('\n✅ Evaluation saved → outputs/evaluation_results.md')

Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

,question,answer_preview,top_source,top_page,top_score,has_fallback
0,What is the main topic discussed in the first ...,The main topic discussed in the first document...,LCFQJGJLCOJ56B3YM3XIPRJ7DFUQPTDG.pdf,96,0.5228,False
1,Summarize the key points of the document colle...,The given documents are intended for a proposa...,APSZJI6LK7RTS54T7IXF4JHQFEOPNCCZ.pdf,5,0.4681,False
2,What specific information is provided about an...,Sources in your Participant Guide for ideas ab...,DOPBKNWBER76WVPP654XCUJGCQEGSOZY.pdf,69,0.5825,False
3,What conclusions or recommendations are made?,The given context contains three sources:\n\n1...,ZYUTCEHUM7TZDJS75XRY72MWSWNR2QPF.pdf,3,0.5927,False
4,"Are there any data, statistics, or numbers men...","Yes, there are several data, statistics, and n...",GF4OGED6DIFR5WHABHTLSTFV7656GKN3.pdf,2,0.5607,False



✅ Evaluation saved → outputs/evaluation_results.md


## 9. Save Sample Results

In [46]:
import json, datetime

sample_result = rag_query(QUESTION, top_k=5)

md_content = f"""# Sample RAG Results

Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

---

## Question
{sample_result['question']}

## Answer
{sample_result['answer']}

## Sources
"""

for src in sample_result['sources']:
    md_content += f"""\n### [{src['source_num']}] {src['filename']} — Page {src['page_number']}
- **Similarity Score:** {src['score']}
- **Snippet:** {src['snippet']}
"""

with open('outputs/sample_results.md', 'w') as f:
    f.write(md_content)

print('✅ Sample results saved → outputs/sample_results.md')
print(md_content[:500])

Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Sample results saved → outputs/sample_results.md
# Sample RAG Results

Generated: 2026-05-31 07:41:26

---

## Question
What is the main topic discussed in the documents?

## Answer
The main topic discussed in the documents is the relevance of documents.

## Sources

### [1] LCFQJGJLCOJ56B3YM3XIPRJ7DFUQPTDG.pdf — Page 96
- **Similarity Score:** 0.6223
- **Snippet:** the relevant 
documents.

### [2] NKJZFXDHGHWNNHGULXM5PSPE7GWWUTR2.pdf — Page 121
- **Similarity Score:** 0.5676
- **Snippet:** - 115 -
 
 
 
 
 
 
 
APPENDIX B - MEMORANDUM OF UND


## 10. Launch Streamlit App via ngrok

In [ ]:
import subprocess                                                                                                                                        
import threading                                                                                                                                         
import time                                                                                                                                              
from pyngrok import ngrok                                                                                                                                

NGROK_TOKEN = 'x'                                                                                      
ngrok.set_auth_token(NGROK_TOKEN)                                                                                                                        

def run_streamlit():                                                                                                                                     
    subprocess.run([                                                                                                                                     
        'streamlit', 'run', 'src/streamlit_app.py',                                                                                                      
        '--server.port', '8501',                                                                                                                         
        '--server.headless', 'true'                                                                                                                      
    ])                                                                                                                                                   

t = threading.Thread(target=run_streamlit, daemon=True)                                                                                                  
t.start()                                                                                                                                                

print("Starting Streamlit...")                                                                                                                           
time.sleep(5)                                                                                                                                            

try:                                                                                                                                                     
    ngrok.kill()                                                                                                                                         
    tunnel = ngrok.connect(8501)                                                                                                                         
    print(f'\n🌐 Streamlit URL Anda berhasil dibuat!')                                                                                                   
    print(f'Buka tautan ini: {tunnel.public_url}')                                                                                                       
except Exception as e:                                                                                                                                   
    print(f'\n❌ Error saat menghubungkan Ngrok: {e}')

Starting Streamlit...



🌐 Streamlit URL Anda berhasil dibuat!
Buka tautan ini: https://ranae-puffy-adrenally.ngrok-free.dev
